In [3]:
import requests
import pandas as pd

In [4]:
import requests
import pandas as pd
import time

# 1. TỪ ĐIỂN MÃ NGÀNH (MAPPING TỪ HTML BẠN CUNG CẤP)
SECTOR_MAP = {
    # 0: "Tất cả", # Bỏ qua cái này để tránh trùng lặp khi quét chi tiết
    345: "Bất động sản & Xây dựng",
    347: "Công nghệ",
    343: "Công nghiệp",
    346: "Dịch vụ",
    # 349: "Doanh nghiệp OTC", # Thường ít giao dịch, có thể bỏ qua nếu chỉ tập trung 3 sàn chính
    # 350: "Doanh nghiệp Upcom", # Đây là sàn, không phải ngành cụ thể
    339: "Hàng tiêu dùng",
    340: "Năng lượng",
    344: "Nguyên vật liệu",
    338: "Nông nghiệp",
    341: "Tài chính", # Ngân hàng, Chứng khoán, Bảo hiểm nằm ở đây
    348: "Viễn thông",
    342: "Y tế"
}

# 2. HÀM LẤY DỮ LIỆU TỪNG NGÀNH
def get_sector_data(sector_id, sector_name, center_id=1):
    """
    Lấy snapshot thị trường của 1 ngành cụ thể.
    center_id: 1 (HOSE). Nếu muốn lấy cả HNX/UPCOM thì phải gọi thêm centerId=2,9
    """
    url = "https://cafef.vn/du-lieu/ajax/mobile/smart/ajaxbandothitruong.ashx"
    
    # center_id=1 là HOSE. Bạn có thể sửa thành vòng lặp [1, 2, 9] để quét cả 3 sàn
    params = {
        "type": 1, 
        "category": sector_id,
        "centerId": center_id 
    }
    
    headers = {"User-Agent": "Mozilla/5.0"}
    
    try:
        r = requests.get(url, params=params, headers=headers)
        data = r.json()
        
        if 'Data' in data and data['Data']:
            df = pd.DataFrame(data['Data'])
            
            # Chọn và đổi tên cột
            cols = {
                'Symbol': 'Ma',
                'Name': 'TenCongTy',
                'Price': 'Gia',
                'ChangePercent': '%ThayDoi',
                'TotalVolume': 'KhoiLuong',
                'MarketCap': 'VonHoa',
                'TotalValue': 'GiaTriGiaoDich'
            }
            # Lọc cột nếu tồn tại
            available_cols = [c for c in cols.keys() if c in df.columns]
            df = df[available_cols].rename(columns=cols)
            
            # QUAN TRỌNG: Gán thêm tên ngành vào DataFrame
            df['Nganh'] = sector_name
            return df
        else:
            return pd.DataFrame()
    except Exception as e:
        print(f"Lỗi lấy ngành {sector_name}: {e}")
        return pd.DataFrame()

# 3. CHẠY QUÉT TOÀN BỘ CÁC NGÀNH (BATCH JOB)
def scan_all_sectors():
    print("🚀 Đang bắt đầu quét toàn bộ các ngành trên sàn HOSE (CenterId=1)...")
    all_dfs = []
    
    for sec_id, sec_name in SECTOR_MAP.items():
        print(f"   -> Đang tải: {sec_name} (ID: {sec_id})...")
        
        df = get_sector_data(sec_id, sec_name, center_id=1)
        
        if not df.empty:
            all_dfs.append(df)
        
        # Nghỉ xíu cho server đỡ chặn
        time.sleep(0.5) 
        
    # Gộp tất cả thành 1 bảng lớn
    if all_dfs:
        master_df = pd.concat(all_dfs, ignore_index=True)
        return master_df
    else:
        return pd.DataFrame()

# --- THỰC THI ---
df_market_snapshot = scan_all_sectors()

print("\n" + "="*40)
print(f"✅ ĐÃ QUÉT XONG! Tổng cộng: {len(df_market_snapshot)} mã cổ phiếu.")
print("="*40)

# Xem mẫu dữ liệu
print(df_market_snapshot[['Ma', 'Gia', '%ThayDoi', 'Nganh']].head(10))

# --- PHÂN TÍCH NHANH (AGGREGATION) ---
if not df_market_snapshot.empty:
    print("\n📊 THỐNG KÊ DÒNG TIỀN THEO NGÀNH (HOSE):")
    # Gom nhóm theo Ngành, tính trung bình % Tăng giảm và Tổng vốn hóa
    sector_stats = df_market_snapshot.groupby('Nganh').agg({
        'Ma': 'count',              # Đếm số mã
        '%ThayDoi': 'mean',         # Trung bình tăng giảm
        'VonHoa': 'sum'             # Tổng vốn hóa
    }).sort_values('%ThayDoi', ascending=False)
    
    # Format lại số liệu cho dễ đọc
    sector_stats['VonHoa (Ty)'] = sector_stats['VonHoa'] / 1_000_000_000
    print(sector_stats[['Ma', '%ThayDoi', 'VonHoa (Ty)']])

🚀 Đang bắt đầu quét toàn bộ các ngành trên sàn HOSE (CenterId=1)...
   -> Đang tải: Bất động sản & Xây dựng (ID: 345)...
   -> Đang tải: Công nghệ (ID: 347)...
   -> Đang tải: Công nghiệp (ID: 343)...
   -> Đang tải: Dịch vụ (ID: 346)...
   -> Đang tải: Hàng tiêu dùng (ID: 339)...
   -> Đang tải: Năng lượng (ID: 340)...
   -> Đang tải: Nguyên vật liệu (ID: 344)...
   -> Đang tải: Nông nghiệp (ID: 338)...
   -> Đang tải: Tài chính (ID: 341)...
   -> Đang tải: Viễn thông (ID: 348)...
   -> Đang tải: Y tế (ID: 342)...

✅ ĐÃ QUÉT XONG! Tổng cộng: 394 mã cổ phiếu.
    Ma    Gia  %ThayDoi                    Nganh
0  KBC  38.00      6.74  Bất động sản & Xây dựng
1  KHG   7.07     -0.42  Bất động sản & Xây dựng
2  DXG  15.70     -1.26  Bất động sản & Xây dựng
3  PDR  17.80      1.71  Bất động sản & Xây dựng
4  VRE  32.00      0.00  Bất động sản & Xây dựng
5  VCG  23.60      1.29  Bất động sản & Xây dựng
6  DLG   3.11      6.87  Bất động sản & Xây dựng
7  DIG  16.25      0.62  Bất động sản & Xâ